# Öğrenilmiş RMS ve centroid kontrolü: küçük spectral CVAE
Bu repo için yazılan koşullu VAE, lisanslı gerçek ayak sesi kaydının 4 saniyelik bölümündeki spektrumları öğrenir. F-RAVE gerçeklenmesi değildir. Tek kayıtla eğitildiği için genel Foley modeli veya ayak sesi kalitesi iddiası yoktur. Çalıştırılmış küçük modelin ağırlıkları `experiments/parametric/spectral_cvae/weights.json` içinde gelir; yeniden eğitim zorunlu değildir. Mevcut PyTorch, NumPy ve soundfile ortamı yeterlidir.

**RMS:** kare içindeki dijital genlik. **Centroid:** FFT magnitude ile ağırlıklı ortalama frekans; parlaklık için öznitelik, pitch değildir. **Latent z:** rastgele örneklenen iki boyutlu gizli değişken. **CVAE:** decoder'a latent ile birlikte kontrol parametrelerini veren VAE. Decoder pozitif spektrum magnitudeleri üretir; sabit random-phase inverse FFT ses oluşturur. Modelin öğrendiği çıktıya RMS/centroid düzeltmesi uygulanmaz. 512 örneklik bloklar ve rastgele faz nedeniyle gürültülü ses ve blok sınırı artefaktları beklenebilir.

In [ ]:
from pathlib import Path
import sys, json
import numpy as np
from IPython.display import Audio, display
ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists(): ROOT = ROOT.parent
assert (ROOT / "src/control").exists(), "Notebooku sound-gen/notebooks altında açın."
sys.path.insert(0, str(ROOT))

In [ ]:
weights = json.loads((ROOT / "experiments/parametric/spectral_cvae/weights.json").read_text())
{"rms_centroid_min": weights["training_control_min"], "rms_centroid_max": weights["training_control_max"], "training_steps": weights["training_steps"]}

Aşağıdaki iki parametreyi değiştirerek üretim yapılır. Girdiler eğitimdeki marjinal aralıklarla sınırlandırılır; iki ayrı aralık içinde olmak birlikte görülen bir kombinasyon olduğunu garanti etmez. Süre tam 512 örneklik bloklara yuvarlanır.

In [ ]:
from src.control.generate import generate
y, sr, metrics = generate("spectral_cvae", ROOT, rms=0.025, centroid_hz=1800, duration_s=2, seed=42, name="local_cvae")
display(Audio(y, rate=sr, normalize=False))
metrics

Kontrolleri karşılaştırırken aynı seed ve aynı playback gain kullanılır. Ayrı normalize edilmiş oynatıcılar şiddet farkını gizleyebilir. Aşağıda ilk çift RMS'i, ikinci çift centroid'i değiştirir.

In [ ]:
clips = []
for name, rms, centroid in [("quiet", .015, 1800), ("loud", .04, 1800), ("dark", .025, 1200), ("bright", .025, 2600)]:
    audio, sr, result = generate("spectral_cvae", ROOT, rms=rms, centroid_hz=centroid, seed=42, name="local_" + name)
    clips.append((name, audio, result))
for name, audio, result in clips:
    print(name, result["actual_mean_block_rms"], result["actual_mean_block_centroid_hz"])
    display(Audio(audio, rate=sr, normalize=False))

Kaydedilen metrikler aynı 512 örneklik bloklardan ölçülür; Hann STFT ile çıkarılan başka centroid tanımlarıyla doğrudan kıyaslanmaz. Üç seed ile kontrol ve diğer parametredeki değişim ölçülür.

In [ ]:
from src.control.spectral_cvae import sweep
report = sweep(ROOT)
report["rms_sweep"], report["centroid_sweep"]

İsteğe bağlı yeniden eğitim: ilk %60 zaman bloğu eğitim, sonraki %20 validation, son %20 geliştirme tanısıdır. Aynı kayıt kullanıldığı ve ilk başarısız deneme sonrası kayıp değiştirildiği için bağımsız test benchmark'ı değildir. KL latent dağılımı düzenler; reconstruction kaybı spektrumu, posterior/prior kontrol kayıpları istenen öznitelikleri öğrenir. CPU'da küçük eğitim çalıştırıldı; Windows yeniden eğitimi burada doğrulanmadı.

In [ ]:
# Yeniden eğitim ağırlıkları ve sonuç dosyalarını değiştirir; çalıştırmak için True yapın.
RETRAIN = False
if RETRAIN:
    from src.control.spectral_cvae import train
    training = train(ROOT, steps=3000)
    report = sweep(ROOT)
else:
    training = json.loads((ROOT / "experiments/parametric/spectral_cvae/training.json").read_text())
training